# Creating a Simple Agent with Tracing

In [2]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [3]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
nutrition_agent = Agent(
    name="NutritionAgent",
    instructions="""You are a helpful nutrition expert. You will provide nutritional advice based on user queries."""
)

Let's execute the Agent:

In [5]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="NutritionAgent", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit with several key benefits, plus a few considerations.
    
    Nutrients in a medium banana (about 118 grams):
    - Potassium: helps blood pressure and heart function
    - Vitamin B6: supports metabolism and brain health
    - Vitamin C: antioxidant and immune support
    - Dietary fiber: aids digestion and satiety
    - Moderate amounts of magnesium, manganese, and some polyphenols
    
    Health benefits:
    - May support heart health and blood pressure
    - Can aid digestion and gut health (fiber and resistant starch in less ripe fruit)
    - Provides quick energy from natural sugars
    
    Things to keep in mind:
    - Sugar content varies with ripeness (ripe bananas are sweeter; unripe have more resistant starch)
    - Moderation for those watching blood sugar or total daily calories
    - People with kidney disease on potassium-restricted diets shoul

Streaming the answer to the screen, token by token

In [6]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Overall, bananas are a healthy, convenient fruit.

Key nutrients (medium banana, ~105 calories):
- Potassium: helps heart and muscle function (about 422 mg, ~12% DV)
- Vitamin B6: supports metabolism and brain health (about 20% DV)
- Vitamin C: antioxidant support (about 10% DV)
- Fiber: aids digestion (about 3 g)
- Carbohydrates: natural sugars plus resistant starch (varying with ripeness)

Benefits:
- Heart and blood pressure support, thanks to potassium
- Steady energy from natural sugars and fiber
- Digestive health from fiber and resistant starch when less ripe

Things to consider:
- Glycemic impact is moderate; pairing with protein/fat can smooth blood sugar
- Very ripe bananas have more sugar; less ripe have more resistant starch
- In kidney disease or when potassium is restricted, monitor intake and consult a clinician
- Rare allergies or severe mace issues are uncommon

Bottom line: bananas are a nutritious, versatile option for most people when eaten in typical portions.

_Good Job!_